# 02 · Tools

In notebook 01 the model invented ShipFast's refund policy and an order status, because it cannot see
either. **Tools** fix that: functions the model can ask your code to run, whose results it then reads.

**You will learn**

- where the tools live, and why they are plain Python functions
- how CrewAI turns a function into a tool
- what a tool call is, and how to run one and hand the result back
- how tool errors and safety checks work when the model gets something wrong

In [1]:
from crew_agents.llm import make_llm  # import crew_agents first: it switches CrewAI's telemetry off

from agent_contracts.harness import verify_model

verify_model()
llm = make_llm()

## The tools are plain functions

ShipFast's five tools live once, in `agent_contracts.shipfast.tools`, shared by all three tutorials. They
know nothing about CrewAI: each takes simple arguments and returns a JSON string. You can call them
directly.

In [2]:
import json

from agent_contracts.shipfast import tools as shipfast

shipfast.reset()  # clears the refund ledger, the tickets and the tool-call log
print([f.__name__ for f in shipfast.TOOLS])
print(json.dumps(json.loads(shipfast.lookup_order("A1001")), indent=2))

['lookup_order', 'track_shipment', 'search_policy', 'issue_refund', 'escalate_to_human']
{
  "order_id": "A1001",
  "customer": "Maria Santos",
  "item": "Wireless Headphones",
  "total": 89.0,
  "status": "in_transit",
  "carrier": "LBC",
  "ordered_on": "2026-10-28",
  "eta": "2026-11-04",
  "delivered_on": null
}


## Turning a function into a tool

CrewAI's `tool(name)` decorator wraps a function as a `BaseTool`: it reads the type hints for the
argument schema and the docstring for the description the model sees. `crew_agents/tools.py` wraps all
five:

In [3]:
import inspect

import crew_agents.tools
from crew_agents.tools import function_schemas, shipfast_tools

print(inspect.getsource(crew_agents.tools._wrap))
tools = shipfast_tools()
for t in tools:
    print(f"{t.name:18} {t.args_schema.model_json_schema()['properties']}")

def _wrap(functions) -> list[BaseTool]:
    return [tool(f.__name__)(f) for f in functions]

lookup_order       {'order_id': {'title': 'Order Id', 'type': 'string'}}
track_shipment     {'order_id': {'title': 'Order Id', 'type': 'string'}}
search_policy      {'query': {'title': 'Query', 'type': 'string'}}
issue_refund       {'order_id': {'title': 'Order Id', 'type': 'string'}, 'amount': {'title': 'Amount', 'type': 'number'}, 'reason': {'title': 'Reason', 'type': 'string'}}
escalate_to_human  {'order_id': {'title': 'Order Id', 'type': 'string'}, 'summary': {'title': 'Summary', 'type': 'string'}}


The docstring is the only instruction the model gets about when to use a tool, so it matters as much
as the code. `issue_refund`'s docstring, for example, says the amount cannot exceed the order total.

## A tool call

Inside a CrewAI agent you never handle tool calls yourself (notebook 03). Calling the `LLM` directly
shows what happens underneath. Pass the tool descriptions in the OpenAI function format
(`function_schemas` converts them) and the model can reply with a **tool call** instead of text: the
name of a tool and the arguments to run it with. It does not run anything itself.

In [4]:
SPECIALIST = ("You are ShipFast's support specialist. Use the tools to check facts before you answer; "
              "never guess order details or policy.")
schemas = function_schemas(tools)

messages = [{"role": "system", "content": SPECIALIST}, {"role": "user", "content": "Hi, where is my order A1001?"}]
calls = llm.call(messages, tools=schemas)
for c in calls:
    print(c.function.name, c.function.arguments, "| id", c.id)

track_shipment {"order_id":"A1001"} | id call_z7hdgkk6


## Running the call and handing back the result

Your code runs the tool, then appends two messages in the OpenAI format: the assistant's tool call, and
a `tool` message carrying the result and the call's id, so the model can match result to request. Ask
again: now the answer is grounded in real data.

In [5]:
tools_by_name = {t.name: t for t in tools}


def run_calls(messages, calls):
    """Run each requested tool and append the assistant and tool messages."""
    messages.append({"role": "assistant", "content": "", "tool_calls": [
        {"id": c.id, "type": "function", "function": {"name": c.function.name, "arguments": c.function.arguments}}
        for c in calls]})
    for c in calls:
        result = tools_by_name[c.function.name].run(**json.loads(c.function.arguments))
        print("tool:", c.function.name, c.function.arguments, "->", result[:100], "...")
        messages.append({"role": "tool", "tool_call_id": c.id, "content": result})


run_calls(messages, calls)
answer = llm.call(messages, tools=schemas)
print("\nanswer:", answer)

tool: track_shipment {"order_id":"A1001"} -> {"order_id": "A1001", "status": "in_transit", "carrier": "LBC", "eta": "2026-11-04", "delivered_on": ...



answer: Your order A1001 is currently in transit with LBC. Here's the latest update:

- **Status:** In Transit
- **Carrier:** LBC
- **Estimated Time of Arrival (ETA):** November 4, 2026

**Tracking Events:**
- October 28, 2026: Order packed at Manila warehouse
- October 30, 2026: Picked up by LBC
- October 31, 2026: Arrived at Cebu sorting hub

If you have any further questions, feel free to ask!


That is the whole idea of an agent, done by hand: call the model, run the tools it asks for, feed the
results back, repeat until it answers in text. Notebook 03 lets a CrewAI `Agent` run that loop.

CrewAI also has a one-step shortcut: pass `available_functions` and `call` runs the requested tool
itself. Note what comes back: the tool's raw output, not an answer. It runs one round, not the loop.

In [6]:
shortcut = llm.call(messages[:2], tools=schemas, available_functions={t.name: t.run for t in tools})
print(type(shortcut).__name__, "->", shortcut[:120], "...")

str -> {"order_id": "A1001", "status": "in_transit", "carrier": "LBC", "eta": "2026-11-04", "delivered_on": null, "tracking": [ ...


## Which tool does it reach for?

Here is the model's **first** move on the opening message of each ShipFast scenario. Nothing is run; we
only look at what it asks for.

In [7]:
from agent_contracts.shipfast.scoring import load_scenarios

seen = set()
for scenario in load_scenarios():
    text = scenario["turns"][0]
    if text in seen:
        continue
    seen.add(text)
    first = llm.call([{"role": "system", "content": SPECIALIST}, {"role": "user", "content": text}], tools=schemas)
    asked = (", ".join(f"{c.function.name}({c.function.arguments})" for c in first)
             if isinstance(first, list) else "(no tool) " + first[:60])
    print(f"{text[:48]:50} -> {asked}")

Hi, where is my order A1001?                       -> track_shipment({"order_id":"A1001"})


My order A1002 never arrived and tracking says i   -> track_shipment({"order_id":"A1002"})


Order A1003 is more than a week late. Please can   -> track_shipment({"order_id":"A1003"})


My vase set from order A1004 arrived broken.       -> track_shipment({"order_id":"A1004"})


I want to return order A1005, the stand wobbles.   -> lookup_order({"order_id":"A1005"})


Where is order A9999?                              -> track_shipment({"order_id":"A9999"})


Can you write me a poem about cats?                -> (no tool) I can't write poems, but I can help you find information or 


Look for two things: whether each choice makes sense for the message, and the out-of-scope request,
where the right move is no tool at all. The model and settings are identical to the LangGraph tutorial,
so its choices should match the ones there; a difference would point to how the framework sends the
request.

## When a tool says no

Tools report problems as data, not exceptions, so the model can read them and recover. An unknown order
comes back as `order_not_found`, and policy P6 says to ask the customer to check the ID.

In [8]:
messages = [{"role": "system", "content": SPECIALIST}, {"role": "user", "content": "Where is order A9999?"}]
for _ in range(3):  # a few rounds of the by-hand loop
    reply = llm.call(messages, tools=schemas)
    if not isinstance(reply, list):
        break
    run_calls(messages, reply)
print("answer:", reply)

tool: track_shipment {"order_id":"A9999"} -> {"error": "order_not_found", "order_id": "A9999"} ...


answer: The order ID "A9999" could not be found. Please double-check the order ID and try again. If you need further assistance, feel free to ask!


## Safety lives in the tool, not the prompt

In one of our early test runs, the model once tried to refund $120 on a $45 order without looking up
the total. The prompt cannot prevent that; the tool can. `issue_refund` refuses any amount above the
order total, whoever calls it.

In [9]:
print(shipfast.issue_refund("A1002", 120, "lost package"))
print(shipfast.issue_refund("A1002", 45, "lost package"))
print("ledger:", shipfast.refunds())

{"error": "amount_exceeds_order_total", "order_id": "A1002", "amount": 120.0, "order_total": 45.0}
{"refund_id": "RF-A1002", "order_id": "A1002", "amount": 45.0, "status": "issued"}
ledger: [{'refund_id': 'RF-A1002', 'order_id': 'A1002', 'amount': 45.0, 'status': 'issued', 'reason': 'lost package'}]


The ledger is what ShipFast's test runner trusts: a result only counts as `refunded` if the ledger
has a matching refund. The model's claim is never enough.

The shared tools also record every call, whichever framework made it. Notebook 07 uses this so all three
tutorials count tool calls the same way.

In [10]:
print(shipfast.tool_calls())
shipfast.reset()

['lookup_order', 'track_shipment', 'track_shipment', 'track_shipment', 'issue_refund', 'issue_refund']


## How the others do this

- **LangGraph** (LangChain) wraps the same functions with `tool()` and attaches them with
  `llm.bind_tools(tools)`; tool calls come back on the reply as `ai.tool_calls`, and invoking a tool with
  the call returns a ready-made `ToolMessage`.
- **Strands** also uses a `@tool` decorator, and its `Agent` runs the call-execute-repeat loop for you
  from the first example.

All three wrap the same five functions from `agent_contracts`, so the tools behave identically; only the
wrapping differs.

## Takeaways

- A tool is a plain function plus a description the model reads; keep the logic outside the framework.
- The model only *asks* for a tool. Your code (or the framework) runs it and hands back the result.
- CrewAI's `available_functions` runs one round of tools; the loop belongs to an `Agent`.
- Errors come back as data the model can act on, and safety checks belong in the tool.
- Next: **03 · Agent**, where CrewAI runs the loop for you.